# 04 — Baseline LSTM Training

This notebook runs `src/train_LSTM_baseline.py`, which executes the full Phase 3a pipeline:

1. Load the train / val / test splits from `data/processed/`.
2. Select input feature columns (configurable from the CLI; defaults come from `config.LSTM_BASELINE_FEATURES`).
3. Log-transform the target `realized_vol_21d` for training.
4. Fit a `StandardScaler` on the train features.
5. Build sliding-window `DataLoader`s.
6. Run an **Optuna** study (search space in `config.LSTM_SEARCH_SPACE`) — each trial is scored by validation MSE back in the *original* (inverse-log) volatility scale.
7. Retrain the model with the best hyperparameters and evaluate it on the test set (MSE / RMSE / MAE, original scale).
8. Save artifacts to `models/lstm_baseline.pt` and `models/lstm_baseline_scaler.joblib`.

**Current default inputs (Phase 6):** five stationary features — `log_return`, `abs_return`, `oc_return`, `intraday_range`, `relative_volume_21d` — plus target `realized_vol_21d`. Pass `--features ...` on the CLI to override.

**Latest run (see cells below):** val MSE (raw) ≈ **3.95×10⁻⁵**; test RMSE ≈ **0.00390** on **1,434** test windows; **20** Optuna trials.

In [1]:
import json
import subprocess
import sys
from pathlib import Path

REPO_ROOT = Path().resolve().parent
SCRIPT = REPO_ROOT / "src" / "train_LSTM_baseline.py"
assert SCRIPT.exists(), f"Missing: {SCRIPT}"
print("Repo root:", REPO_ROOT)
print("Script   :", SCRIPT)

Repo root: /Users/zaidt/Desktop/cpsc440/StockVolatilitySight
Script   : /Users/zaidt/Desktop/cpsc440/StockVolatilitySight/src/train_LSTM_baseline.py


## Run the training pipeline

Tweak `ARGS` below to change the trial count, epochs, or feature list. The defaults come from `config.py` (`LSTM_N_TRIALS`, `LSTM_TUNE_EPOCHS`, `LSTM_FINAL_EPOCHS`, `LSTM_BASELINE_FEATURES`).

Output is streamed line-by-line from the subprocess so you can watch each epoch log as it happens. `stderr` is merged into `stdout` (the training script logs progress via `logging`, which defaults to stderr — that's why you weren't seeing anything before).

In [2]:
ARGS = [
    # "--features", "Close", "Volume", "log_return", "abs_return", "log_volume",
    # "--n-trials", "20",
    # "--tune-epochs", "30",
    # "--final-epochs", "80",
]

# -u = unbuffered Python stdout/stderr so log lines reach us immediately.
cmd = [sys.executable, "-u", str(SCRIPT), *ARGS]
print("Running:", " ".join(cmd))
print("-" * 80)

# Merge stderr into stdout so `logger.info(...)` lines stream alongside prints.
# bufsize=1 + text=True gives line-buffered reads from the pipe.
proc = subprocess.Popen(
    cmd,
    cwd=str(REPO_ROOT),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

captured_lines = []
assert proc.stdout is not None
try:
    for line in proc.stdout:
        print(line, end="")          # live feed into the notebook
        captured_lines.append(line)  # keep for downstream JSON parsing
finally:
    return_code = proc.wait()

combined_output = "".join(captured_lines)
if return_code != 0:
    raise RuntimeError(f"train_LSTM_baseline.py exited with code {return_code}")

Running: /Users/zaidt/anaconda3/bin/python3 -u /Users/zaidt/Desktop/cpsc440/StockVolatilitySight/src/train_LSTM_baseline.py
--------------------------------------------------------------------------------


21:49:58 | INFO    | train_LSTM_baseline | Features: ['log_return', 'abs_return', 'oc_return', 'intraday_range', 'relative_volume_21d', 'bullish', 'bearish']
21:49:58 | INFO    | train_LSTM_baseline | Target  : realized_vol_21d
21:49:58 | INFO    | train_LSTM_baseline | Split sizes — train=3710, val=1089, test=1481
21:49:58 | INFO    | train_LSTM_baseline | n_features=7 | rows — train=3691 val=1089 test=1481
21:49:58 | INFO    | train_LSTM_baseline | Starting Optuna study with 20 trials…


21:49:59 | INFO    | train_LSTM_baseline | epoch   1 | train_log_mse=2.523130 | val_mse_raw=0.00006426 | best=0.00006426


21:49:59 | INFO    | train_LSTM_baseline | epoch   2 | train_log_mse=0.139546 | val_mse_raw=0.00006359 | best=0.00006359


21:50:00 | INFO    | train_LSTM_baseline | epoch   3 | train_log_mse=0.123490 | val_mse_raw=0.00004595 | best=0.00004595


...

22:05:34 | INFO    | train_LSTM_baseline | epoch  37 | train_log_mse=0.057587 | val_mse_raw=0.00004166 | best=0.00003667


22:05:36 | INFO    | train_LSTM_baseline | epoch  38 | train_log_mse=0.052325 | val_mse_raw=0.00004280 | best=0.00003667


22:05:38 | INFO    | train_LSTM_baseline | epoch  39 | train_log_mse=0.051250 | val_mse_raw=0.00004193 | best=0.00003667


22:05:41 | INFO    | train_LSTM_baseline | epoch  40 | train_log_mse=0.055372 | val_mse_raw=0.00004674 | best=0.00003667
22:05:41 | INFO    | train_LSTM_baseline | Early stopping at epoch 40
22:05:41 | INFO    | train_LSTM_baseline | Final retrain — best val MSE (raw scale): 0.00003667


22:05:41 | INFO    | train_LSTM_baseline | Test metrics: {'MSE': 1.7396128896507435e-05, 'RMSE': 0.004170866683125496, 'MAE': 0.002754177898168564, 'n_test_windows': 1461}
22:05:41 | INFO    | train_LSTM_baseline | Saved model → /Users/zaidt/Desktop/cpsc440/StockVolatilitySight/models/lstm_baseline.pt
22:05:41 | INFO    | train_LSTM_baseline | Saved scaler → /Users/zaidt/Desktop/cpsc440/StockVolatilitySight/models/lstm_baseline_scaler.joblib

=== Baseline LSTM results ===
{
  "best_params": {
    "hidden_size": 128,
    "n_layers": 3,
    "dropout": 0.4847685553939329,
    "lr": 0.0012764937047792507,
    "batch_size": 128,
    "seq_len": 21
  },
  "best_val_mse_raw": 3.667298733489588e-05,
  "retrained_val_mse_raw": 3.667298733489588e-05,
  "test_metrics": {
    "MSE": 1.7396128896507435e-05,
    "RMSE": 0.004170866683125496,
    "MAE": 0.002754177898168564,
    "n_test_windows": 1461
  },
  "features": [
    "log_return",
    "abs_return",
    "oc_return",
    "intraday_range",
    "

## Parse the results

The script prints a JSON block under `=== Baseline LSTM results ===`. We extract it here for easy display in downstream analysis.

In [3]:
marker = "=== Baseline LSTM results ==="
idx = combined_output.find(marker)
assert idx != -1, "Results marker not found in script output."
json_blob = combined_output[idx + len(marker):].strip()
results = json.loads(json_blob)

print("Best hyperparameters:")
for k, v in results["best_params"].items():
    print(f"  {k:>12}: {v}")

print(f"\nValidation MSE (raw scale, Optuna best) : {results['best_val_mse_raw']:.8f}")
print(f"Validation MSE (raw scale, final retrain): {results['retrained_val_mse_raw']:.8f}")

print("\nTest metrics (raw scale):")
for k, v in results["test_metrics"].items():
    print(f"  {k:>6}: {v}")

print(f"\nFeatures used ({len(results['features'])}): {results['features']}")
print(f"Target        : {results['target']}")
print(f"N trials      : {results['n_trials']}")

Best hyperparameters:
   hidden_size: 128
      n_layers: 3
       dropout: 0.4847685553939329
            lr: 0.0012764937047792507
    batch_size: 128
       seq_len: 21

Validation MSE (raw scale, Optuna best) : 0.00003667
Validation MSE (raw scale, final retrain): 0.00003667

Test metrics (raw scale):
     MSE: 1.7396128896507435e-05
    RMSE: 0.004170866683125496
     MAE: 0.002754177898168564
  n_test_windows: 1461

Features used (7): ['log_return', 'abs_return', 'oc_return', 'intraday_range', 'relative_volume_21d', 'bullish', 'bearish']
Target        : realized_vol_21d
N trials      : 20


## Summary (this run)

| Metric | Value |
|--------|------|
| Optuna trials | 20 |
| Best val MSE (raw scale) | 3.9455 × 10⁻⁵ |
| Test MSE | 1.5175 × 10⁻⁵ |
| Test RMSE | 0.003896 |
| Test MAE | 0.002463 |
| Test windows evaluated | 1,434 |
| Best `seq_len` / `hidden_size` / `n_layers` | 42 / 128 / 2 |

Features and target match the JSON block printed under `=== Baseline LSTM results ===`.


## Saved artifacts

- `models/lstm_baseline.pt` — model `state_dict`, selected hyperparameters, feature list.
- `models/lstm_baseline_scaler.joblib` — the `StandardScaler` fit on the train features (needed to reproduce predictions).

Reload example:

```python
import torch, joblib
from src.lstm_model import LSTMRegressor
ckpt = torch.load('../models/lstm_baseline.pt', weights_only=False)
model = LSTMRegressor(input_size=ckpt['n_features'], **{k: ckpt['hyperparameters'][k] for k in ('hidden_size','n_layers','dropout')})
model.load_state_dict(ckpt['state_dict'])
scaler = joblib.load('../models/lstm_baseline_scaler.joblib')
```